# Fase 0 — Preparación de datos

**Procesamiento y Clasificación de Datos · MCD, FCFM-UANL**

Este notebook se corre **una sola vez**. Descarga las reseñas, toma una muestra manejable y
balanceada, y guarda un solo archivo que usan las Tareas 1, 2 y 3.

## Datos

**Amazon Reviews 2023** (McAuley Lab, UCSD). Reseñas reales de Amazon con calificación de
1 a 5 estrellas. Se usan dos categorías:

| Categoría | Reseñas totales | Por qué |
|---|---:|---|
| `Digital_Music` | 130 mil | Quien reseña música habla de gustos y emociones |
| `All_Beauty` | 701 mil | Quien reseña productos de belleza habla de resultados |

Ese contraste es lo que se analiza en la Tarea 1, y la categoría sirve como etiqueta de
clasificación en la Tarea 3. La calificación en estrellas es la etiqueta de la Tarea 2.

## Por qué se toma una muestra

Dos razones:

1. **Tamaño.** Los archivos completos suman ~830 mil reseñas. Para tareas de curso, con 50 mil
   por categoría sobra, y todo corre en segundos en vez de minutos.
2. **Balance.** En Amazon la mayoría de las reseñas son de 5 estrellas. Si se muestrea al azar,
   un clasificador aprende a decir "5 estrellas" siempre y parece bueno sin serlo. Por eso el
   muestreo es **estratificado**: la misma cantidad de reseñas por cada estrella.

Cita: Hou et al. (2024), *Bridging Language and Items for Retrieval and Recommendation*.

## 1. Descarga

Correr en la terminal, dentro de la carpeta `data/` del repo (está en el `.gitignore`, así que
nada de esto se sube a GitHub):

```bash
cd data
curl -L -O https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Digital_Music.jsonl.gz
curl -L -O https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/All_Beauty.jsonl.gz
```

Son ~35 MB y ~180 MB. El formato es JSONL: un JSON por línea, comprimido con gzip.
Pandas los lee directo, sin descomprimir.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")
CATEGORIAS = {
    "musica": "Digital_Music.jsonl.gz",
    "belleza": "All_Beauty.jsonl.gz",
}
POR_ESTRELLA = 10_000     # reseñas por cada estrella -> 50,000 por categoría
SEMILLA = 42

for archivo in CATEGORIAS.values():
    ruta = DATA / archivo
    assert ruta.exists(), f"Falta {ruta} — revisa el paso de descarga"
    print(f"{archivo}: {ruta.stat().st_size / 1e6:.0f} MB")

Digital_Music.jsonl.gz: 26 MB
All_Beauty.jsonl.gz: 94 MB


## 2. Carga

Solo se conservan las columnas que las tareas usan. `images` y los IDs de producto/usuario no
sirven para análisis de texto.

In [3]:
COLUMNAS = ["rating", "title", "text", "helpful_vote", "verified_purchase", "timestamp"]

crudos = {}
for etiqueta, archivo in CATEGORIAS.items():
    df = pd.read_json(DATA / archivo, lines=True)
    df = df[COLUMNAS].copy()
    df["categoria"] = etiqueta
    crudos[etiqueta] = df
    print(f"{etiqueta}: {len(df):,} reseñas")

musica: 130,434 reseñas
belleza: 701,528 reseñas


## 3. Limpieza mínima

Tres filtros, cada uno con su razón:

1. **Texto vacío o de una palabra** — no hay nada que analizar en "Good." repetido mil veces.
   Se pide un mínimo de 3 palabras.
2. **Rating fuera de 1–5** — por si el archivo trae basura.
3. **Duplicados exactos de texto** — Amazon tiene reseñas copy-paste (bots, importaciones).
   Un texto repetido 500 veces distorsionaría todas las frecuencias de la Tarea 1.

In [4]:
def limpiar(df):
    n0 = len(df)
    df = df.dropna(subset=["text", "rating"])
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"].str.split().str.len() >= 3]
    df = df[df["rating"].between(1, 5)]
    df = df.drop_duplicates(subset="text")
    print(f"  {n0:,} -> {len(df):,}  (se quitó {100 * (1 - len(df) / n0):.1f}%)")
    return df


limpios = {}
for etiqueta, df in crudos.items():
    print(etiqueta)
    limpios[etiqueta] = limpiar(df)

musica
  130,434 -> 114,416  (se quitó 12.3%)
belleza
  701,528 -> 631,483  (se quitó 10.0%)


## 4. Muestreo estratificado

De cada categoría: 10,000 reseñas por cada estrella (1★ a 5★), elegidas al azar con semilla
fija para que el resultado sea reproducible.

Si alguna estrella tiene menos de 10,000 disponibles, se toman todas las que haya — por eso
al final se imprime la tabla real, para saber exactamente con qué quedamos.

In [5]:
def muestrear(df, por_estrella, semilla):
    partes = []
    for estrella, grupo in df.groupby("rating"):
        n = min(por_estrella, len(grupo))
        partes.append(grupo.sample(n=n, random_state=semilla))
    return pd.concat(partes, ignore_index=True)


muestras = []
for etiqueta, df in limpios.items():
    m = muestrear(df, POR_ESTRELLA, SEMILLA)
    muestras.append(m)
    print(f"{etiqueta}: {len(m):,} reseñas en la muestra")

datos = pd.concat(muestras, ignore_index=True)

# mezclar para que las categorías no queden en bloques
datos = datos.sample(frac=1, random_state=SEMILLA).reset_index(drop=True)

print(f"\nTotal: {len(datos):,} reseñas")
pd.crosstab(datos["categoria"], datos["rating"])

musica: 34,750 reseñas en la muestra
belleza: 50,000 reseñas en la muestra

Total: 84,750 reseñas


rating,1,2,3,4,5
categoria,,,,,
belleza,10000,10000,10000,10000,10000
musica,5691,3016,6043,10000,10000


La tabla de arriba es importante para los reportes: dice cuántas reseñas reales hay por
categoría y estrella. Si alguna celda quedó por debajo de 10,000, ahí está documentado.

## 5. Guardar

In [6]:
SALIDA = DATA / "resenas_muestra.csv"
datos.to_csv(SALIDA, index=False)
print(f"Guardado: {SALIDA}  ({SALIDA.stat().st_size / 1e6:.0f} MB)")

# verificación de ida y vuelta
check = pd.read_csv(SALIDA)
assert len(check) == len(datos)
print("Verificado: el archivo se relee sin pérdida")
check.head(3)

Guardado: ../data/resenas_muestra.csv  (32 MB)
Verificado: el archivo se relee sin pérdida


,rating,title,text,helpful_vote,verified_purchase,timestamp,categoria
0,1,Does not dry! It's been over 3 months!,Mixture is off and does not dry even if your t...,0,True,2021-10-19 19:42:32.032,belleza
1,2,Horrible hair,I really dislike it,0,True,2021-04-19 17:17:01.118,belleza
2,5,The quality of Wilkerson,"This cd, is fantastic, my first never heard of...",0,True,2020-11-05 20:22:53.348,musica


## Listo

Las tres tareas empiezan igual:

```python
import pandas as pd
datos = pd.read_csv("../data/resenas_muestra.csv")
```

| Columna | Uso |
|---|---|
| `text`, `title` | el texto a analizar |
| `categoria` | las "dos fuentes" (Tarea 1) y etiqueta de tema (Tarea 3) |
| `rating` | comparación con sentimiento (Tarea 2) y etiqueta (Tarea 3) |
| `verified_purchase`, `helpful_vote` | contexto, por si algún análisis los pide |